# SIMBES — Módulo 6: Diagnóstico DIFA / API RP 11S1

**Objetivo:** Implementar el motor de diagnóstico DIFA para el sistema BES:  
1. Librería de patrones de falla con pesos de síntomas  
2. Algoritmo de matching síntoma → patrón  
3. Visualización de resultados y distribución por serie API  
4. Análisis de sensibilidad del motor de diagnóstico  

---
**Estándar:** API RP 11S1 — Recommended Practice for Electrical Submersible Pump Teardown Report  
**Series:** 3700 (Corrosión) · 4900 (Sello primario) · 5400 (Sellos secundarios) · 5900 (Cable/eléctrico)

In [ ]:
import sys
import os
import json
sys.path.insert(0, os.path.join('..', 'backend'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# Estilo SIMBES dark
plt.rcParams.update({
    'figure.facecolor': '#0B0F1A',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#1E293B',
    'axes.labelcolor':  '#CBD5E1',
    'xtick.color':      '#64748B',
    'ytick.color':      '#64748B',
    'text.color':       '#CBD5E1',
    'grid.color':       '#1E293B',
    'grid.linestyle':   '--',
    'legend.facecolor': '#111827',
    'legend.edgecolor': '#1E293B',
    'font.family':      'monospace',
})

# Cargar librería de fallas
with open('../frontend/src/data/failure-library.json', 'r', encoding='utf-8') as f:
    LIBRARY = json.load(f)

PATTERNS = LIBRARY['failure_patterns']
print(f"Patrones cargados: {len(PATTERNS)}")
for p in PATTERNS:
    print(f"  {p['api_code']} — {p['title'][:55]}")

## 1. Motor de Diagnóstico — Algoritmo de Matching

### Lógica

Dado un conjunto de síntomas activos $S$, la confianza para el patrón $P_k$ se calcula como:

$$
\text{score}_k = \sum_{i \in S} w_{ki}
$$

$$
\text{maxScore}_k = \sum_{\text{all } i} w_{ki}
$$

$$
\text{penalty}_k = 0.15 \times |S \cap C_k|
$$

$$
\text{confidence}_k = \max\left(0,\ \frac{\text{score}_k}{\text{maxScore}_k} - \text{penalty}_k\right)
$$

Donde $w_{ki}$ es el peso del síntoma $i$ para el patrón $k$, y $C_k$ son los síntomas que contradicen el patrón $k$.

Solo se muestran patrones con $\text{confidence} > 0.10$.

In [ ]:
def diagnose(active_symptoms: set, patterns=PATTERNS) -> list:
    """
    Motor de diagnóstico DIFA.
    
    Retorna patrones ordenados por confianza descendente.
    Solo incluye patrones con confidence > 0.10.
    
    @ref API RP 11S1 §5 — Failure Pattern Classification
    """
    if not active_symptoms:
        return []
    
    results = []
    for p in patterns:
        weights     = p.get('symptom_weights', {})
        contradicts = set(p.get('contradicts', []))
        
        score     = sum(w for s, w in weights.items() if s in active_symptoms)
        max_score = sum(weights.values())
        
        n_contra  = len(contradicts & active_symptoms)
        penalty   = 0.15 * n_contra
        
        confidence = max(0.0, score / max_score - penalty) if max_score > 0 else 0.0
        
        results.append({
            **p,
            'score':        score,
            'max_score':    max_score,
            'confidence':   confidence,
            'n_contra':     n_contra,
        })
    
    return sorted(
        [r for r in results if r['confidence'] > 0.10],
        key=lambda x: x['confidence'],
        reverse=True
    )

# ── Test con síntomas del caso Pozo Delfín-3 ──────────────────────
sintomas_delfin3 = {'lowCurrent', 'gradualFlow', 'highDischPress'}
resultado = diagnose(sintomas_delfin3)

print(f"Síntomas: {sintomas_delfin3}")
print(f"Patrones identificados: {len(resultado)}\n")
print(f"{'Código':<8} {'Confianza':>10} {'Severidad':<12} {'Título':<40}")
print("-" * 75)
for r in resultado:
    print(f"{r['api_code']:<8} {r['confidence']:>9.1%}  {r['severity']:<12} {r['title'][:40]}")

In [ ]:
# Visualización: confianza de los patrones para Delfín-3
SEVERITY_COLORS = {
    'critical': '#DC2626',
    'high':     '#EF4444',
    'medium':   '#F59E0B',
    'low':      '#22C55E',
}

fig, ax = plt.subplots(figsize=(10, 4))

names  = [f"{r['api_code']}\n{r['title'].split('→')[0].strip()[:25]}" for r in resultado]
confs  = [r['confidence'] * 100 for r in resultado]
colors = [SEVERITY_COLORS.get(r['severity'], '#64748B') for r in resultado]

bars = ax.barh(names, confs, color=colors, alpha=0.85, height=0.55)
ax.axvline(75, color='#22C55E', lw=1.5, ls='--', alpha=0.7, label='Alta confianza (75%)')
ax.axvline(45, color='#F59E0B', lw=1,   ls='--', alpha=0.7, label='Confianza media (45%)')

for bar, conf in zip(bars, confs):
    ax.text(conf + 1, bar.get_y() + bar.get_height()/2,
            f'{conf:.0f}%', va='center', fontsize=9, color='#CBD5E1')

ax.set_xlabel('Confianza [%]')
ax.set_title('Motor DIFA — Pozo Delfín-3\nSíntomas: lowCurrent + gradualFlow + highDischPress', pad=10)
ax.set_xlim(0, 115)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 2. Análisis de Casos Representativos

Prueba del motor con las 4 combinaciones de síntomas más frecuentes en campo.

In [ ]:
CASOS = [
    {
        'nombre': 'Gas Lock',
        'sintomas': {'erraticCurrent', 'abruptFlow', 'lowDischPress', 'highMotorTemp'},
        'expected_code': '4910',
    },
    {
        'nombre': 'Incrustación de escala',
        'sintomas': {'gradualFlow', 'highCurrent', 'highDischPress'},
        'expected_code': '3720',
    },
    {
        'nombre': 'Falla de sello (H₂S)',
        'sintomas': {'lowIR', 'erraticCurrent'},
        'expected_code': '3730',
    },
    {
        'nombre': 'Surging severo',
        'sintomas': {'erraticCurrent', 'highVibration', 'highCurrent'},
        'expected_code': '5410',
    },
    {
        'nombre': 'Falla de rodamiento',
        'sintomas': {'highVibration', 'gradualFlow'},
        'expected_code': '5430',
    },
]

print(f"{'Caso':<25} {'Síntomas':>8} {'#1 Código':>12} {'Confianza':>10} {'Esperado':>10} {'OK?':>5}")
print("-" * 75)

for caso in CASOS:
    res = diagnose(caso['sintomas'])
    if res:
        top = res[0]
        ok  = '✓' if top['api_code'] == caso['expected_code'] else '✗'
        print(f"{caso['nombre']:<25} {len(caso['sintomas']):>8}  {top['api_code']:>12}  {top['confidence']:>9.1%}  {caso['expected_code']:>10}  {ok:>5}")
    else:
        print(f"{caso['nombre']:<25} {len(caso['sintomas']):>8}  {'No match':>12}  {'—':>10}  {caso['expected_code']:>10}  ✗")

In [ ]:
# Gráfica comparativa de todos los casos
fig, axes = plt.subplots(1, len(CASOS), figsize=(15, 4), sharey=False)

for ax, caso in zip(axes, CASOS):
    res = diagnose(caso['sintomas'])[:4]  # Top 4
    if not res:
        ax.text(0.5, 0.5, 'Sin match', ha='center', va='center', color='#64748B')
        ax.set_title(caso['nombre'], fontsize=8, pad=6)
        continue
    
    names  = [r['api_code'] for r in res]
    confs  = [r['confidence'] * 100 for r in res]
    colors = [SEVERITY_COLORS.get(r['severity'], '#64748B') for r in res]
    
    bars = ax.bar(names, confs, color=colors, alpha=0.85, width=0.5)
    ax.axhline(75, color='#22C55E', lw=1, ls='--', alpha=0.5)
    ax.set_title(caso['nombre'], fontsize=8, pad=6, color='#CBD5E1')
    ax.set_ylim(0, 110)
    ax.set_ylabel('%' if ax == axes[0] else '')
    ax.tick_params(axis='x', labelsize=7, colors='#64748B')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, conf in zip(bars, confs):
        ax.text(bar.get_x() + bar.get_width()/2, conf + 2,
                f'{conf:.0f}%', ha='center', fontsize=7, color='#CBD5E1')

plt.suptitle('Motor DIFA — Confianza por Caso', color='#CBD5E1', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 3. Árbol de Diagnóstico Visual — Síntoma → Serie API

Mapa de calor: ¿qué síntomas activan qué series API?

In [ ]:
# Todos los síntomas disponibles
ALL_SYMPTOMS = [
    'lowCurrent', 'highCurrent', 'erraticCurrent', 'lowIR',
    'gradualFlow', 'abruptFlow', 'highVibration',
    'highMotorTemp', 'highDischPress', 'lowDischPress',
]

# Para cada síntoma individual: ¿qué patrones activa con qué confianza?
API_SERIES_COLORS = {
    '3700': '#F59E0B',
    '4900': '#EF4444',
    '5400': '#A78BFA',
    '5900': '#38BDF8',
}

# Matriz síntoma × patrón (confianza)
pattern_codes = [p['api_code'] for p in PATTERNS]
matrix = np.zeros((len(ALL_SYMPTOMS), len(PATTERNS)))

for i, sym in enumerate(ALL_SYMPTOMS):
    results = diagnose({sym})
    for r in results:
        j = next((k for k, p in enumerate(PATTERNS) if p['api_code'] == r['api_code']), None)
        if j is not None:
            matrix[i, j] = r['confidence']

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(range(len(PATTERNS)))
ax.set_xticklabels([f"{p['api_code']}" for p in PATTERNS], rotation=30, ha='right', fontsize=8)
ax.set_yticks(range(len(ALL_SYMPTOMS)))
ax.set_yticklabels(ALL_SYMPTOMS, fontsize=9)

# Anotaciones
for i in range(len(ALL_SYMPTOMS)):
    for j in range(len(PATTERNS)):
        if matrix[i, j] > 0:
            ax.text(j, i, f'{matrix[i,j]:.0%}', ha='center', va='center',
                   fontsize=7, color='black' if matrix[i,j] > 0.5 else 'white')

plt.colorbar(im, ax=ax, label='Confianza (síntoma individual)', fraction=0.02)
ax.set_title('Mapa Síntoma × Patrón — Motor DIFA\n(cada celda = confianza con síntoma único)', pad=12)
plt.tight_layout()
plt.show()

## 4. Análisis de Sensibilidad — Efecto de Añadir Síntomas

¿Cómo cambia la confianza del patrón #1 al ir añadiendo síntomas progresivamente?

In [ ]:
# Caso Gas Lock: síntomas progresivos
GAS_LOCK_SYMPTOMS_ORDERED = [
    ('abruptFlow',    'Caudal abrupto'),
    ('erraticCurrent','Corriente errática'),
    ('lowDischPress', 'P descarga baja'),
    ('highMotorTemp', 'T motor alta'),
    ('lowCurrent',    'Corriente baja'),
]

active = set()
steps  = []

for sym_id, sym_label in GAS_LOCK_SYMPTOMS_ORDERED:
    active.add(sym_id)
    res = diagnose(active)
    top = res[0] if res else None
    steps.append({
        'n_symptoms': len(active),
        'added':      sym_label,
        'top_code':   top['api_code'] if top else '—',
        'confidence': top['confidence'] if top else 0,
    })
    
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Evolución de confianza
x     = [s['n_symptoms'] for s in steps]
confs = [s['confidence'] * 100 for s in steps]

ax1.plot(x, confs, 'o-', color='#EF4444', lw=2.5, markersize=8, markerfacecolor='#0B0F1A', markeredgewidth=2)
ax1.axhline(75, color='#22C55E', lw=1.5, ls='--', alpha=0.7, label='Alta confianza')
ax1.axhline(45, color='#F59E0B', lw=1,   ls='--', alpha=0.7, label='Media')
for i, s in enumerate(steps):
    ax1.annotate(s['added'], (s['n_symptoms'], s['confidence']*100),
                 xytext=(5, 5), textcoords='offset points',
                 fontsize=7, color='#CBD5E1')
ax1.set_xlabel('Número de síntomas seleccionados')
ax1.set_ylabel('Confianza patrón #1 [%]')
ax1.set_title('Gas Lock — Confianza vs N° síntomas', pad=10)
ax1.set_xlim(0.5, 5.5)
ax1.set_ylim(0, 110)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

# Distribución de series API en el resultado final
final_result = diagnose(active)
series_count = {}
for r in final_result:
    s = r['api_series']
    series_count[s] = series_count.get(s, 0) + 1

series_labels = [f"Serie {k}\n({v} patrón{'es' if v>1 else ''})" for k, v in series_count.items()]
series_vals   = list(series_count.values())
series_colors = [API_SERIES_COLORS.get(k, '#64748B') for k in series_count.keys()]

ax2.pie(series_vals, labels=series_labels, colors=series_colors, autopct='%1.0f%%',
        textprops={'fontsize': 8, 'color': '#CBD5E1'},
        wedgeprops={'edgecolor': '#0B0F1A', 'linewidth': 2})
ax2.set_title('Distribución de Series API — Gas Lock', pad=10)

plt.suptitle('Análisis de Sensibilidad del Motor DIFA', color='#CBD5E1', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

print("\nEvolución paso a paso:")
for s in steps:
    print(f"  +{s['added']:<25} → #{s['top_code']}  confianza={s['confidence']:.0%}")

## 5. Resumen — Codificación API RP 11S1

| Serie | Categoría | Códigos clave | Causa tipo |
|-------|-----------|---------------|------------|
| 3700 | Corrosión / Picadura | 3712, 3720, 3730 | Recirculación, escala, H₂S |
| 4900 | Sello primario | 4910, 4930 | Gas lock, elastómero |
| 5400 | Sellos secundarios | 5410, 5430 | Surging, rodamiento |
| 5900 | Terciarios / Cable | 5910, 5930 | Caída voltaje, H₂S cable |

**Regla de oro:** el código que determina la causa raíz es el del **primer daño** — todos los demás son consecuencias.

---
*Notebook generado automáticamente por SIMBES para el Módulo 6 — Diagnóstico DIFA.*  
*Versión: 1.0 · Fecha: 2026-03-15*